### Анализ тональности

#### Подключение библиотек

In [1]:
import pandas as pd
import numpy as np

#### Загрузка данных

In [2]:
df = pd.read_csv('Restaurant reviews.csv')
df.shape

(10000, 8)

In [3]:
df.head()

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures,7514
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0,2447.0
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0,NaN
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0,NaN
3,Beyond Flavours,Swapnil Sarkar,Soumen das and Arun was a great guy. Only beca...,5,"1 Review , 1 Follower",5/24/2019 22:11,0,NaN
4,Beyond Flavours,Dileep,Food is good.we ordered Kodi drumsticks and ba...,5,"3 Reviews , 2 Followers",5/24/2019 21:37,0,NaN


In [4]:
df.columns

Index(['Restaurant', 'Reviewer', 'Review', 'Rating', 'Metadata', 'Time',
       'Pictures', '7514'],
      dtype='str')

In [6]:
df = df.drop(['Restaurant', 'Reviewer', 'Metadata', 'Time', 'Pictures', '7514'], axis = 1)

In [7]:
df.head()

,Review,Rating
0,"The ambience was good, food was quite good . h...",5
1,Ambience is too good for a pleasant evening. S...,5
2,A must try.. great food great ambience. Thnx f...,5
3,Soumen das and Arun was a great guy. Only beca...,5
4,Food is good.we ordered Kodi drumsticks and ba...,5


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Review  9955 non-null   str  
 1   Rating  9962 non-null   str  
dtypes: str(2)
memory usage: 2.8 MB


In [9]:
df['Rating'].unique()

<ArrowStringArray>
['5', '4', '1', '3', '2', '3.5', '4.5', '2.5', '1.5', 'Like', nan]
Length: 11, dtype: str

In [10]:
df['Rating'].value_counts()

Rating
5       3832
4       2373
1       1735
3       1193
2        684
4.5       69
3.5       47
2.5       19
1.5        9
Like       1
Name: count, dtype: int64

In [11]:
df['Rating'] = np.where(df['Rating'] == 'Like', df['Rating'].value_counts().idxmax(), df['Rating'])

In [12]:
df['Rating'].value_counts()

Rating
5      3833
4      2373
1      1735
3      1193
2       684
4.5      69
3.5      47
2.5      19
1.5       9
Name: count, dtype: int64

In [13]:
df['Rating'] = list(map(lambda data: float(data) > 3, df['Rating']))
df['Rating'].value_counts()

Rating
True     6322
False    3678
Name: count, dtype: int64

#### NLP

In [14]:
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import re

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [20]:
ps = PorterStemmer()

In [15]:
df['Review'].isnull().sum()

np.int64(45)

In [16]:
df['Review'] = df['Review'].fillna('Nothing')

In [17]:
reviewList = list(map(lambda data: word_tokenize(data), df['Review']))

In [18]:
reviewList[0]

['The',
 'ambience',
 'was',
 'good',
 ',',
 'food',
 'was',
 'quite',
 'good',
 '.',
 'had',
 'Saturday',
 'lunch',
 ',',
 'which',
 'was',
 'cost',
 'effective',
 '.',
 'Good',
 'place',
 'for',
 'a',
 'sate',
 'brunch',
 '.',
 'One',
 'can',
 'also',
 'chill',
 'with',
 'friends',
 'and',
 'or',
 'parents',
 '.',
 'Waiter',
 'Soumen',
 'Das',
 'was',
 'really',
 'courteous',
 'and',
 'helpful',
 '.']

In [21]:
for index, word_list in enumerate(reviewList):
    reviewList[index] = ' '.join([ps.stem(str(word)) for word in word_list if not word in stopwords.words('english') and word not in ['.', ',', '?', '!', '@', '$', '/'] and not word.isspace()])
    

In [25]:
reviewList[0]

'the ambienc good food quit good saturday lunch cost effect good place sate brunch one also chill friend parent waiter soumen da realli courteou help'

In [36]:
emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002500-\U00002BEF" 
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u2640-\u2642" 
        u"\u2600-\u2B55"
        u"\u200d"
        u"\u23cf"
        u"\u23e9"
        u"\u231a"
        u"\ufe0f"
        u"\u3030"
                      "]+", re.UNICODE)

In [35]:
print(u"\U0001F629")

😩


In [38]:
regexPunct = r"[\s\w\d]"

In [39]:
for index, word_list in enumerate(reviewList):
    reviewList[index] = re.sub(emoji_pattern, r'', word_list)
    reviewList[index] = ''.join(re.findall(regexPunct, reviewList[index], re.MULTILINE))
    reviewList[index] = ' '.join(reviewList[index].split())

In [40]:
df['Review'] = reviewList
df.head()

,Review,Rating
0,the ambienc good food quit good saturday lunch...,True
1,ambienc good pleasant even servic prompt food ...,True
2,a must tri great food great ambienc thnx servi...,True
3,soumen da arun great guy onli behavior sincere...,True
4,food goodw order kodi drumstick basket mutton ...,True


#### Разделение данных

In [41]:
X = df['Review']
y = df['Rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

#### Построение модели

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

In [47]:
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifire', SVC())
])

In [48]:
pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)

0.8775

In [51]:
pipeline_2 = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1,3))),
    ('classifire', RandomForestClassifier(n_estimators=300))
])

In [52]:
pipeline_2.fit(X_train, y_train)
pipeline_2.score(X_test, y_test)

0.86

In [53]:
positive_sentence = "I absolutely love this product! It has exceeded all my expectations. It's well-designed, durable, and incredibly useful. I can't imagine my life without it now. I would highly recommend it to anyone looking for a high-quality, reliable product"
negative_sentence = "I bought this product, and it was a complete waste of money. It stopped working after just a week, and the customer service was unhelpful and rude. I will never purchase anything from this company again."
sentence = "I purchased this product, and it has been nothing but a disappointment. The quality is abysmal, and it didn't even last a week before falling apart. It's clear that the manufacturer prioritized cutting costs over producing a reliable item. I would strongly advise against buying this product – it's a complete waste of money."
sentence_2 = "I hate this product"
sentence_3 = " "

In [54]:
pipeline.predict([positive_sentence, negative_sentence, sentence, sentence_2, sentence_3])

array([ True, False,  True,  True,  True])